In [1]:
import pandas as pd
import numpy as np
from sympy import false

In [28]:
df=pd.read_parquet("loan_information.parquet")
df.head()

,zonename,pmuk_id,shamity_id,loan_component,loan_product_id,member_id,loan_id,disbursed_date,loan_amount,total_interest,...,paid_principal,paid_interest,kisti_paid_date,kisti_status,pre_sthiti,pre_bokeya_sthiti,post_sthiti,post_bokeya_sthiti,principal,service_charge
0,Cumilla Zone,16,18723,JAGORON,41,1204302,1903973,2025-09-09,80000.00,10800.00,...,6696,904,2026-02-01,paid,58800,0,52800,0,5286,714
1,Cumilla Zone,16,18723,JAGORON,41,1204302,1903973,2025-09-09,80000.00,10800.00,...,6696,904,2026-03-03,paid,52800,0,50800,0,1762,238
2,Cumilla Zone,16,24262,LIVELIHOOD IMPROVEMENT LOAN,48,879050,1336295,2024-04-04,24000.00,3240.00,...,2010,270,2026-02-19,paid,18740,18740,18240,18240,441,59
3,Dhaka North Zone,19,9031,JAGORON,41,1030179,1543189,2024-11-17,50000.00,6750.00,...,4185,565,2026-03-08,paid,20750,20750,19750,19750,881,119
4,Cumilla Zone,16,19694,AGROSOR,22,1181043,1854277,2025-07-27,300000.00,40500.00,...,25110,3390,2026-02-01,paid,198000,0,169500,0,25110,3390


In [12]:
df.groupby('loan_id')['overdue'].sum().loc[1508685]

np.int64(382400)

In [14]:
df[df['loan_id'].isin([1508685])]

,zonename,pmuk_id,shamity_id,loan_component,loan_product_id,member_id,loan_id,disbursed_date,loan_amount,total_interest,...,paid_principal,paid_interest,kisti_paid_date,kisti_status,pre_sthiti,pre_bokeya_sthiti,post_sthiti,post_bokeya_sthiti,principal,service_charge
37,Dinajpur Zone,09,12345,AGROSOR,22,836175,1508685,2024-10-20,400000.00,54000.00,...,33479,4521,2026-02-01,paid,116600,116600,112000,112000,4053,547
38,Dinajpur Zone,09,12345,AGROSOR,22,836175,1508685,2024-10-20,400000.00,54000.00,...,14449,1951,2026-04-30,bokeya,112000,112000,108600,108600,2996,404
1063553,Dinajpur Zone,09,12345,AGROSOR,22,836175,1508685,2024-10-20,400000.00,54000.00,...,14449,1951,2026-04-30,bokeya,108600,108600,100600,100600,7048,952
3364206,Dinajpur Zone,09,12345,AGROSOR,22,836175,1508685,2024-10-20,400000.00,54000.00,...,14449,1951,2026-04-30,bokeya,100600,100600,95600,95600,4405,595


In [16]:
df.groupby(['zonename','loan_id'])['overdue'].first().sum()

np.int64(2770694603)

In [17]:
df.groupby(['zonename', 'loan_id'])['overdue'].first().groupby('zonename').sum()

zonename
B-Baria Zone                148180511
Barishal Zone               156248900
Bhola Zone                  100213483
Bogura Zone                  55900080
Chattogram North Zone       141532626
Chattogram South Zone       146409387
Cumilla Zone                166137056
Dhaka North Zone            180316789
Dhaka South Zone            201864640
Dinajpur Zone               168347310
Faridpur Zone               185824650
Jashore Zone                198052633
Kishoregonj Zone             69602415
Kushtia Zone                118504237
Mymensingh Zone             124506769
Parbotto Chottogram Zone    133723898
Patuakhali Zone             144456955
Rajshahi Zone               183442528
Rangpur Zone                147429736
Name: overdue, dtype: int64

In [20]:
df['kisti_amount'].sum()

np.int64(17511960959)

In [27]:
df[df['shamity_id']==226]

,zonename,pmuk_id,shamity_id,loan_component,loan_product_id,member_id,loan_id,disbursed_date,loan_amount,total_interest,...,paid_principal,paid_interest,kisti_paid_date,kisti_status,pre_sthiti,pre_bokeya_sthiti,post_sthiti,post_bokeya_sthiti,principal,service_charge
1623886,Dhaka South Zone,01,226,JAGORON,25,7620,5706,2020-03-05,100000.00,13000.00,...,103,14,2026-03-10,bokeya,10433,10433,10383,10383,44,6


In [ ]:
monthly = (
        df.groupby("month")
        .agg(
            total_due       = ("kisti_amount",   "sum"),
            total_collected = ("paid_amount",     "sum"),
            total_principal = ("paid_principal",  "sum"),
            total_interest  = ("paid_interest",   "sum"),
            avg_delay       = ("delay_days",      "mean"),
            loan_count      = ("loan_id",         "nunique"),
        )
        .assign(collection_rate=lambda x: safe_div(x["total_collected"], x["total_due"]) * 100)
        .reset_index()
    )
monthly["month_str"] = monthly["month"].astype(str)

print(monthly.to_string(index=False))

In [ ]:
import polars as  pl
start = datetime(2026, 2, 1)
end = datetime(2026, 4, 30)
os.makedirs('loans_data_DSZ', exist_ok =True)

current_start = start

while current_start <= end:

    current_end = min(current_start + timedelta(days=5), end + timedelta(days=1))

    print(f"Fetching {current_start} -> {current_end}")

    df = pl.read_database(
        query,
        conn,
        execute_options={
            "params": (
                current_start,
                current_end,
                current_start,
                current_end,
            )
        }
    )
    file_name = current_end.strftime("%Y-%m-%d")
    df.write_parquet(f"loans_data_DSZ/{file_name}.parquet")
    del df
    gc.collect()
    current_start = current_end
print('done')